In [ ]:
import json
import pandas as pd
import numpy as np

In [ ]:
# Load raw flood indicator fields from VCP
vcp_path = "data_sources/hazards/VCP_Tracts.geojson"
with open(vcp_path) as f:
    vcp_data = json.load(f)

records = []
for feat in vcp_data["features"]:
    props = feat["properties"]
    records.append({
        "GEOID": str(props["GEOID"]),
        "flood_bam_100_pct": props.get("Flood_BAM_100_pct"),
        "flood_bam_500_pct": props.get("Flood_BAM_500_pct"),
        "flood_verywet_pre_pct": props.get("Flood_verywet_pre_pct"),
        "flood_verywet_fut_pct": props.get("Flood_verywet_fut_pct"),
    })

df = pd.DataFrame(records)
print(f"{len(df)} tracts loaded")
df.describe()

In [ ]:
# BAM values are percentages (0–100) representing the share of LandScan squares
# within each tract that fall inside the floodplain. VCP does not re-normalize BAM
# before summing because it is already on a 0–1 scale. Divide by 100 to restore that.
df["bam_100"] = df["flood_bam_100_pct"] / 100
df["bam_500"] = df["flood_bam_500_pct"] / 100

In [ ]:
# Normalize very-wet-days percentages using min-max across BOTH periods combined.
# VCP: "When there are two periods involved, now and projected, the scales for the
# two values are combined before normalization. The maximum is the largest value in
# either period and the minimum is the lowest value in either period."
verywet_all = pd.concat([df["flood_verywet_pre_pct"], df["flood_verywet_fut_pct"]], ignore_index=True)
vw_min = verywet_all.min()
vw_max = verywet_all.max()
print(f"Very wet days range across both periods: {vw_min:.4f} – {vw_max:.4f}")

df["verywet_pre_norm"] = (df["flood_verywet_pre_pct"] - vw_min) / (vw_max - vw_min)
df["verywet_fut_norm"] = (df["flood_verywet_fut_pct"] - vw_min) / (vw_max - vw_min)

In [ ]:
# Compute raw composite flood scores following VCP methodology:
#   score = BAM (0–1, not re-normalized) + 0.5 * verywet_norm (0–1)
# VCP: "because flood risk is primarily driven by local topography and regional
# drainage, we feel that the BAM data should have an overwhelming influence on our
# flood risk score. To ensure that it does, we reduce the weight of this variable
# [very wet days] by half."
df["flood_hazard_raw"]     = df["bam_100"] + 0.5 * df["verywet_pre_norm"]
df["flood_hazard_fut_raw"] = df["bam_500"] + 0.5 * df["verywet_fut_norm"]

In [ ]:
# Normalize each index to 0–100 using min-max (consistent with heat_hazard_idx_norm)
def normalize_0_100(series):
    s_min = series.min()
    s_max = series.max()
    return ((series - s_min) / (s_max - s_min)) * 100

df["flood_hazard_idx_norm"]     = normalize_0_100(df["flood_hazard_raw"])
df["flood_hazard_fut_idx_norm"] = normalize_0_100(df["flood_hazard_fut_raw"])

df[["flood_hazard_idx_norm", "flood_hazard_fut_idx_norm"]].describe()

In [ ]:
output = df[[
    "GEOID",
    "flood_bam_100_pct",
    "flood_bam_500_pct",
    "flood_verywet_pre_pct",
    "flood_verywet_fut_pct",
    "flood_hazard_idx_norm",
    "flood_hazard_fut_idx_norm",
]]
output.to_csv("data/flood_hazard.csv", index=False)
print(f"Wrote {len(output)} rows → data/flood_hazard.csv")
output.head()